In [1]:
import numpy as np 
import matplotlib.pyplot as plt 
import pandas as pd

plt.rcParams.update({'mathtext.fontset': 'cm'})
plt.rcParams.update({'font.family': 'STIXGeneral'})
plt.rcParams.update({'font.size': 20})
plt.rcParams.update({'axes.xmargin': 0})

In [3]:
def load_zi_lockin_by_timejump(filepath, jump_threshold=None):
    import numpy as np
    import pandas as pd

    # Read file, skip comments
    with open(filepath, 'r') as f:
        lines = [l.strip() for l in f if not l.startswith('%') and l.strip()]

    # Parse numeric data
    data = np.array([list(map(float, line.split(';'))) for line in lines])
    time = data[:, 0]
    amp = data[:, 1]

    # Compute time differences
    dt = np.diff(time)

    # Detect jump
    if jump_threshold is None:
        jump_idx = np.where(dt < 0)[0]
    else:
        jump_idx = np.where(np.abs(dt) > jump_threshold)[0]

    # ---------------- NO JUMP CASE ----------------
    if len(jump_idx) == 0:
        time = time - time[0]

        df = pd.DataFrame({
            'time': time,
            'amp': amp
        })

        return df

    # ---------------- JUMP CASE ----------------
    split = jump_idx[0] + 1

    t1, a1 = time[:split], amp[:split]
    t2, a2 = time[split:], amp[split:]

    t1 = t1 - t1[0]
    t2 = t2 - t2[0]

    df = pd.DataFrame({
        'time_ch1': pd.Series(t1),
        'amp_ch1': pd.Series(a1),
        'time_ch2': pd.Series(t2),
        'amp_ch2': pd.Series(a2),
    })

    return df

In [ ]:
path = ".\meas_plotter_20260423_094913.txt"

df_long = load_zi_lockin_by_timejump(path)
df_long

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

# --- normalize signal to [0, 1] ---
y = df_long["amp"].to_numpy()
I_norm_long = (y - y.min()) / (y.max() - y.min())
I_norm_long = I_norm_long*0.85 + 0.15
t_long = df_long["time"].to_numpy()

plt.figure()
plt.plot(t_long, I_norm_long)
plt.show()

In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import get_window

def get_fft(x, fs, window='hann', nfft=None, detrend=True):
    x = np.asarray(x)

    # --- remove DC / trend ---
    if detrend:
        x = x - np.mean(x)

    N = len(x)

    # --- windowing ---
    if window is not None:
        w = get_window(window, N)
        xw = x * w
        scale = np.sum(w) / N
    else:
        xw = x
        scale = 1.0

    # --- zero padding (optional) ---
    if nfft is None:
        nfft = N
    elif nfft < N:
        raise ValueError("nfft must be >= len(x)")

    # --- FFT (real → positive freqs only) ---
    X = np.fft.rfft(xw, n=nfft)
    freqs = np.fft.rfftfreq(nfft, d=1/fs)

    # --- amplitude spectrum ---
    amplitude = np.abs(X) / (N * scale)

    # correct for single-sided spectrum
    amplitude[1:-1] *= 2

    # --- plot ---
    # plt.figure(figsize=(8, 4))
    # plt.plot(freqs, amplitude)
    # plt.xlabel("Frequency (Hz)")
    # plt.ylabel("Amplitude")
    # plt.title("FFT (single-sided)")
    # plt.grid(True)
    # plt.tight_layout()
    # plt.show()

    return freqs, amplitude

In [ ]:
freqs, amplitude = get_fft(x = I_norm_long, fs = 1/(np.average(np.diff(t_long))))

plt.figure()
plt.plot(freqs, amplitude)
plt.xlim(0, 5e3)
plt.show()

In [ ]:
from scipy.signal import savgol_filter

I_smooth = savgol_filter(I_norm_long, 25, 3)

N_downsample = 1

fs = 1/(np.average(np.diff(t_long)))  # Hz
f_downsample = fs/N_downsample

t = t_long[::N_downsample]
x = I_smooth[::N_downsample]

plt.figure()
plt.plot(t_long, I_norm_long, linewidth = 1)
plt.plot(t, x, linewidth = 1)
plt.xlim(0, 0.025)
plt.show()

### Logistic map

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial import KDTree

# 1. Generate Chaotic Data (Logistic Map)
def generate_logistic_map(r, x0, n):
    x = np.zeros(n)
    x[0] = x0
    for i in range(1, n):
        x[i] = r * x[i-1] * (1 - x[i-1])
    return x

n_points = 2000
data = generate_logistic_map(3.9, 0.5, n_points)

# 2. Algorithm: Tracking Divergence of Nearest Neighbors
def get_lyapunov_data(data, delay=1, embed_dim=2, max_t=30):
    n = len(data)
    num_vectors = n - (embed_dim - 1) * delay
    # Reconstruct phase space
    space = np.array([data[i : i + embed_dim * delay : delay] for i in range(num_vectors)])
    tree = KDTree(space)
    
    divergence = np.zeros(max_t)
    counts = np.zeros(max_t)
    
    # We sample points to track how they move away from their neighbors
    valid_range = num_vectors - max_t
    indices = np.random.choice(range(valid_range), min(800, valid_range), replace=False)

    for i in indices:
        # Find the nearest neighbor (avoiding points close in time)
        dist, idxs = tree.query(space[i], k=15)
        neighbor_idx = -1
        for d, j in zip(dist, idxs):
            if abs(i - j) > 10: # Theiler window
                neighbor_idx = j
                break
        
        if neighbor_idx == -1 or neighbor_idx >= valid_range: continue

        # Track divergence over time t
        for t in range(max_t):
            d_t = np.linalg.norm(space[i + t] - space[neighbor_idx + t])
            if d_t > 0:
                divergence[t] += np.log(d_t)
                counts[t] += 1

    return divergence / (counts + 1e-10), space

avg_log_div, phase_space = get_lyapunov_data(data)

# 3. Plotting
plt.figure(figsize=(15, 5))

# Plot A: Time Series
plt.subplot(1, 3, 1)
plt.plot(data[:], 'b-', lw=1)
plt.title("Time Series")

# Plot B: Phase Space (The Attractor)
plt.subplot(1, 3, 2)
plt.scatter(phase_space[:, 0], phase_space[:, 1], s=1, color='purple', alpha=0.5)
plt.title("Phase Space Attractor")
plt.xlabel("$x(t)$"); plt.ylabel("$x(t+1)$")

# Plot C: The Lyapunov Divergence Plot
plt.subplot(1, 3, 3)
time_axis = np.arange(len(avg_log_div))
plt.plot(time_axis[1:], avg_log_div[1:], 'ro-', markersize=3, label='Avg Log Separation')

# Fit slope to the linear growth phase
slope, intercept = np.polyfit(time_axis[1:12], avg_log_div[1:12], 1)
plt.plot(time_axis[1:12], slope * time_axis[1:12] + intercept, 'k--', label=f'LE ($\lambda$) ≈ {slope:.3f}')

plt.title("Divergence vs. Time")
plt.xlabel("Time Step ($t$)"); plt.ylabel("Avg Log Distance")
plt.legend()
plt.tight_layout()
plt.show()

### Our data

In [38]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KDTree
from scipy.signal import welch
from scipy.spatial.distance import cdist

# -------------------------------
# 1. Time-delay embedding
# -------------------------------
def delay_embedding(x, dim, tau):
    N = len(x)
    M = N - (dim - 1) * tau
    return np.array([x[i:i + dim*tau:tau] for i in range(M)])

# -------------------------------
# 2. Estimate delay (tau)
# using autocorrelation (simple)
# -------------------------------
def estimate_delay(x, fs):
    x = x - np.mean(x)
    autocorr = np.correlate(x, x, mode='full')
    autocorr = autocorr[len(autocorr)//2:]
    autocorr /= autocorr[0]

    # first minimum or 1/e crossing
    for i in range(1, len(autocorr)):
        if autocorr[i] < 1/np.e:
            return i
    return 10  # fallback

# -------------------------------
# 3. Estimate embedding dimension
# (very simple false nearest neighbors proxy)
# -------------------------------
def estimate_dim(x, tau, max_dim=10):
    from sklearn.neighbors import NearestNeighbors

    fnn_ratios = []

    for dim in range(1, max_dim):
        Y1 = delay_embedding(x, dim, tau)
        Y2 = delay_embedding(x, dim+1, tau)

        # --- FIX: align lengths ---
        min_len = min(len(Y1), len(Y2))
        Y1 = Y1[:min_len]
        Y2 = Y2[:min_len]

        nbrs = NearestNeighbors(n_neighbors=2).fit(Y1)
        distances, indices = nbrs.kneighbors(Y1)

        R = distances[:, 1]
        neighbor_idx = indices[:, 1]

        # safe indexing now
        dist_next = np.abs(Y2[:, -1] - Y2[neighbor_idx, -1])

        fnn = np.mean(dist_next / (R + 1e-10) > 10)
        fnn_ratios.append(fnn)

    return np.argmin(fnn_ratios) + 1

# -------------------------------
# 4. Lyapunov exponent (Rosenstein)
# -------------------------------
def lyapunov_rosenstein(x, fs, dim, tau, max_t=200, theiler=50):
    Y = delay_embedding(x, dim, tau)
    N = len(Y)

    tree = KDTree(Y)

    neighbors = []
    for i in range(N):
        dist, ind = tree.query(Y[i].reshape(1, -1), k=10)
        for j in ind[0][1:]:
            if abs(i - j) > theiler:
                neighbors.append((i, j))
                break

    neighbors = np.array(neighbors)

    divergence = np.zeros(max_t)

    for k in range(max_t):
        vals = []
        for i, j in neighbors:
            if i + k < N and j + k < N:
                d = np.linalg.norm(Y[i + k] - Y[j + k])
                if d > 0:
                    vals.append(np.log(d))
        if len(vals) > 0:
            divergence[k] = np.mean(vals)
        else:
            divergence[k] = np.nan

    t = np.arange(max_t) / fs

    # fit linear region
    fit_start = 5
    fit_end = -1#int(max_t * 0.3)

    coeffs = np.polyfit(t[fit_start:fit_end], divergence[fit_start:fit_end], 1)
    lle = coeffs[0]

    return t, divergence, lle

# -------------------------------
# 5. Visualization: attractor
# -------------------------------
def plot_attractor(x, tau):
    plt.figure(figsize=(6,6))
    plt.plot(x[:-2*tau], x[tau:-tau], alpha=0.5, linewidth = 1)
    plt.xlabel("x(t)")
    plt.ylabel("x(t+tau)")
    plt.title("Reconstructed attractor (2D)")
    plt.show()

# -------------------------------
# MAIN PIPELINE
# -------------------------------
def analyze_timeseries(x, fs, tau = None, dim = None, max_t=200, theiler=50):
    if tau == None:
        tau = estimate_delay(x, fs)
        print(f"Estimated tau: {tau}")

    if dim == None:
        dim = estimate_dim(x, tau)
        print(f"Estimated dim: {dim}")

    t, div, lle = lyapunov_rosenstein(x, fs, dim, tau, max_t=max_t, theiler=theiler)

    print(f"Largest Lyapunov exponent: {lle:.4f} 1/s")

    # plot divergence
    plt.figure()
    plt.plot(t, div)
    plt.xlabel("Time (s)")
    plt.ylabel("log distance")
    plt.title("Lyapunov divergence")
    plt.show()

    # attractor
    plot_attractor(x, tau)

    return lle

In [ ]:
dominant_frequency = 2e3
period_dominant = 1/dominant_frequency
N_points_per_period = f_downsample/dominant_frequency

tau = int(N_points_per_period/5) #heuristic

print(f"f_downsample = {f_downsample/1e3:.2f} kHz")
print(f"Points per dominant period: {N_points_per_period:.2f}")
print(f"Estimated tau: {tau}")

In [14]:
# x_test = x[:10000]
# for tau in [1, 3, 5, 10, 20, 50, 100, 250, 1000]:
#     plt.figure()
#     plt.plot(x_test[:-tau], x_test[tau:], alpha=0.5, linewidth = 0.1)
#     plt.title(tau)

### Parmater choices

- $\tau$: Tau determines the spacing between points in the embedded vector. It should capture a "significant" fraction of the dominant frequencies in the signal. If too short: you only capture noise. If too long: you will start capturing next oscillation periods

- dim: This is the total size of the embedded vector. How high dimensionality your attractor is. Typically 2 - 6 or so.

- max_t: The max amount of time for calculating the distance. Should be a few oscillation periods.

- theiler window: should not pick two vectors that are close in time so they should be separated at least one oscillation period.

In [ ]:
taus = [2, 5, 10]
dims = [3, 5, 8]

for tau in taus:
    for dim in dims:
        print(f"tau = {tau}, dim = {dim}")
        lle = analyze_timeseries(
            x, 
            f_downsample, 
            tau = 5, 
            dim = 5, 
            max_t = int(3*N_points_per_period), 
            theiler=int(N_points_per_period))

In [ ]:
N = 15
sigma = 60
i_alpha = -1
i_delta = 175

raw_filename = f'C:\\Users\\lion_remote\\Documents\\Geert\\results\\sweep_results_N={N}_sigma={sigma}.npz'

# =====================================================================
# DATA DISCOVERY PIPELINE
# =====================================================================
print(f'loading data from {raw_filename} ...')

data = np.load(raw_filename)

alpha = data['alphas'][i_alpha]
delta = data['deltas'][i_delta]
print(f'Indexed at alpha={alpha}, delta={delta}')

amp = np.sum(data['states'][i_alpha, i_delta, :N, :], axis=0)
time = data['time']
print(f'{len(amp)} timesteps loaded')

print('data succesfully loaded')

In [ ]:
plt.figure()
plt.plot(time, amp)
plt.xlim(1000, 1050)
plt.show()

In [ ]:
f_downsample = 1
dominant_frequency = 2
period_dominant = 1/dominant_frequency
N_points_per_period = f_downsample/dominant_frequency
print(np.array(amp))

lle = analyze_timeseries(
            np.array(amp), 
            f_downsample, 
            tau = None, 
            dim = None, 
            max_t = int(3*N_points_per_period), 
            theiler=int(N_points_per_period))

In [42]:
import numpy as np

def phase_space_reconstruction(time_series, emb_dim, delay):
    """Reconstructs phase space using time-delay embedding."""
    n = len(time_series)
    M = n - (emb_dim - 1) * delay
    X = np.array([time_series[i : i + emb_dim * delay : delay] for i in range(M)])
    return X

def calculate_lle(time_series, emb_dim=3, delay=1, theiler_window=10, max_steps=20):
    """
    Calculates the Largest Lyapunov Exponent (LLE) using Rosenstein's algorithm.
    """
    X = phase_space_reconstruction(time_series, emb_dim, delay)
    M = len(X)
    
    # Step 1: Find nearest neighbors for each point outside the Theiler window
    nn_indices = np.zeros(M, dtype=int)
    for i in range(M):
        min_dist = float('inf')
        nn_idx = -1
        for j in range(M):
            if abs(i - j) > theiler_window:
                dist = np.linalg.norm(X[i] - X[j])
                if 0 < dist < min_dist:
                    min_dist = dist
                    nn_idx = j
        nn_indices[i] = nn_idx

    # Step 2: Track divergence over time steps
    divergence = np.zeros(max_steps)
    counts = np.zeros(max_steps)
    
    for i in range(M):
        j = nn_indices[i]
        if j == -1:
            continue
        for k in range(max_steps):
            if i + k < M and j + k < M:
                d = np.linalg.norm(X[i + k] - X[j + k])
                if d > 0:
                    divergence[k] += np.log(d)
                    counts[k] += 1
                    
    # Step 3: Compute average log-divergence
    avg_div = np.zeros(max_steps)
    valid_steps = []
    for k in range(max_steps):
        if counts[k] > 0:
            avg_div[k] = divergence[k] / counts[k]
            valid_steps.append(k)
            
    # Step 4: Fit a straight line to get the slope (Lyapunov Exponent)
    if len(valid_steps) > 1:
        slope, _ = np.polyfit(valid_steps, avg_div[valid_steps], 1)
        return slope
    else:
        raise ValueError("Not enough data points or valid steps to calculate LLE.")




In [ ]:
try:
    # Adjust parameters depending on your dataset's frequency and dynamics
    lle = calculate_lle(amp, emb_dim=3, delay=1, theiler_window=2, max_steps=3)
    print(f"Calculated Largest Lyapunov Exponent: {lle:.5f}")
except ValueError as e:
    print(e)

In [ ]:
import nolds

lle = nolds.lyap_r(data, emb_dim=3, lag=1)